# 04. 학습용 feature selection

이 노트북은 03번에서 만든 넓은 window dataset을 05번 baseline 모델이 바로 사용할 수 있는 학습 입력 계약으로 정리한다.

핵심 기준:
- feature 선택 통계는 strict train split에서만 계산한다.
- strict / relaxed 학습 후보셋을 분리해서 저장한다.
- 결측 대체값은 strict train split 기준으로만 계산한다.
- 기본 `trainable_windows.csv`는 strict + imputed 버전으로 둔다.


In [1]:

from pathlib import Path
import re

import pandas as pd

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 220)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('Cannot find pyproject.toml from current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ml_windows' / 'ml_window_dataset.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_features'

DEFAULT_TRAINABLE_WINDOWS_PATH = OUTPUT_DIR / 'trainable_windows.csv'
STRICT_RAW_PATH = OUTPUT_DIR / 'trainable_windows_strict.csv'
RELAXED_RAW_PATH = OUTPUT_DIR / 'trainable_windows_relaxed.csv'
STRICT_IMPUTED_PATH = OUTPUT_DIR / 'trainable_windows_strict_imputed.csv'
RELAXED_IMPUTED_PATH = OUTPUT_DIR / 'trainable_windows_relaxed_imputed.csv'
FEATURE_COLUMNS_PATH = OUTPUT_DIR / 'feature_columns.csv'
METADATA_COLUMNS_PATH = OUTPUT_DIR / 'metadata_columns.csv'
IMPUTATION_VALUES_PATH = OUTPUT_DIR / 'imputation_values.csv'
CATEGORICAL_FEATURE_MAP_PATH = OUTPUT_DIR / 'categorical_feature_map.csv'
FEATURE_FAMILY_SUMMARY_PATH = OUTPUT_DIR / 'feature_family_summary.csv'

MAX_FEATURE_MISSING_RATE = 0.50
BASELINE_LABELS = {'normal', 'pre_fault'}
FEATURE_SELECTION_SPLIT_COLUMN = 'split_regime_based'
FEATURE_SELECTION_SPLIT_VALUE = 'train'
REQUIRE_NON_NULL_IN_BOTH_MANUFACTURERS = True
MAX_ONE_HOT_CARDINALITY = 12

METADATA_ROLES = {
    'manufacturer': 'identifier',
    'substation_id': 'identifier',
    'source_file': 'source_trace',
    'window_start': 'time_range',
    'window_end': 'time_range',
    'main_missing_sensors': 'explanation_text',
    'main_changed_sensors': 'explanation_text',
    'label': 'target',
    'fault_label': 'target_context',
    'fault_event_id': 'event_identifier',
    'estimated_lead_time_hours': 'lead_time_target',
    'normal_event_related': 'label_context',
    'maintenance_related': 'interpretation_context',
    'disturbance_count': 'interpretation_context',
    'leakage_blocked_fault_count': 'leakage_guard',
    'window_source_type': 'training_control',
    'use_for_supervised_training': 'training_control',
    'normal_reference_outlier': 'training_control',
    'normal_reference_outlier_count': 'training_control',
    'normal_reference_filter_reason': 'training_control',
    'configuration_type': 'configuration_context',
    'normal_reference_group': 'configuration_context',
    'season_bucket': 'time_context',
    'split_time_based': 'evaluation_split',
    'split_substation_based': 'evaluation_split',
    'split_regime_based': 'evaluation_split',
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'?? ??: {INPUT_PATH}')
print(f'?? ??: {OUTPUT_DIR}')
print(f'feature ?? ?? split: {FEATURE_SELECTION_SPLIT_COLUMN}={FEATURE_SELECTION_SPLIT_VALUE}')
print(f'??? ??: {MAX_FEATURE_MISSING_RATE:.0%}')


?? ??: C:\Project3\HeatGrid_Agent\data\processed\ml_windows\ml_window_dataset.csv
?? ??: C:\Project3\HeatGrid_Agent\data\processed\ml_features
feature ?? ?? split: split_regime_based=train
??? ??: 50%


In [2]:
ml_window_dataset = pd.read_csv(INPUT_PATH)

print('전체 데이터셋 크기:', ml_window_dataset.shape)
display(ml_window_dataset[['label', 'window_source_type', 'use_for_supervised_training']].head())
display(ml_window_dataset['label'].value_counts(dropna=False).rename_axis('라벨').reset_index(name='행 수'))


전체 데이터셋 크기: (3270, 252)


,label,window_source_type,use_for_supervised_training
0,normal,normal_context,True
1,normal,normal_context,True
2,normal,normal_context,True
3,normal,normal_context,True
4,normal,normal_context,True


,라벨,행 수
0,normal,1818
1,pre_fault,815
2,unlabeled,637


## 1. strict / relaxed 학습 후보셋 생성

- `strict`: 지도학습 후보 중 `data_quality_issue == False`인 행
- `relaxed`: `data_quality_issue` 여부와 무관한 전체 지도학습 후보
- feature 선택 통계는 `strict + split_time_based == train`에서만 계산한다.


In [3]:

for column in ml_window_dataset.columns:
    if column.endswith('__dominant'):
        METADATA_ROLES[column] = 'control_context'

eligible_label_mask = ml_window_dataset['label'].isin(BASELINE_LABELS)
quality_strict_mask = eligible_label_mask & (ml_window_dataset['data_quality_issue'] == False)
selection_mask = quality_strict_mask & (ml_window_dataset['use_for_supervised_training'] == True)

strict_windows = ml_window_dataset.loc[quality_strict_mask].copy()
relaxed_windows = ml_window_dataset.loc[eligible_label_mask].copy()
selection_stats_windows = ml_window_dataset.loc[
    selection_mask & (ml_window_dataset[FEATURE_SELECTION_SPLIT_COLUMN] == FEATURE_SELECTION_SPLIT_VALUE)
].copy()

if selection_stats_windows.empty:
    raise ValueError('Feature selection statistics dataset is empty. Check split configuration.')

scenario_summary = pd.DataFrame([
    {
        'dataset': 'strict',
        'row_count': len(strict_windows),
        'normal': int((strict_windows['label'] == 'normal').sum()),
        'pre_fault': int((strict_windows['label'] == 'pre_fault').sum()),
    },
    {
        'dataset': 'relaxed',
        'row_count': len(relaxed_windows),
        'normal': int((relaxed_windows['label'] == 'normal').sum()),
        'pre_fault': int((relaxed_windows['label'] == 'pre_fault').sum()),
    },
    {
        'dataset': 'selection_stats_train_only',
        'row_count': len(selection_stats_windows),
        'normal': int((selection_stats_windows['label'] == 'normal').sum()),
        'pre_fault': int((selection_stats_windows['label'] == 'pre_fault').sum()),
    },
])

display(scenario_summary)
display(pd.crosstab(selection_stats_windows['manufacturer'], selection_stats_windows['label']))


,dataset,row_count,normal,pre_fault
0,strict,2526,1767,759
1,relaxed,2633,1818,815
2,selection_stats_train_only,1668,1224,444


label,normal,pre_fault
manufacturer,,
manufacturer 1,667,203
manufacturer 2,557,241


## 2. metadata 컬럼 정의

식별자, 시간 범위, 라벨, 설명용 문자열, split 컬럼은 baseline feature에서 제외하고 metadata로 보존한다.


In [4]:
metadata_columns = [column for column in strict_windows.columns if column in METADATA_ROLES]

metadata_columns_table = pd.DataFrame({
    'column_name': metadata_columns,
    'role': [METADATA_ROLES[column] for column in metadata_columns],
    'dtype': [str(strict_windows[column].dtype) for column in metadata_columns],
})

display(metadata_columns_table)


,column_name,role,dtype
0,manufacturer,identifier,str
1,substation_id,identifier,int64
2,source_file,source_trace,str
3,window_start,time_range,str
4,window_end,time_range,str
5,main_missing_sensors,explanation_text,str
6,main_changed_sensors,explanation_text,str
7,season_bucket,time_context,str
8,label,target,str
9,fault_label,target_context,str


## 3. strict train split 기준 baseline feature 선택

선택 규칙:
- 숫자형 또는 bool형만 사용
- strict train split 기준 결측률 50% 이하
- strict train split에서 상수 컬럼 제외
- 양 제조사 모두에서 non-null 값이 존재하는 컬럼만 선택


In [5]:

def sanitize_feature_name(value: str) -> str:
    value = re.sub(r'[^0-9A-Za-z?-?]+', '_', str(value).strip().lower())
    value = re.sub(r'_+', '_', value).strip('_')
    return value or 'missing'


def detect_feature_family(column_name: str, derived_categorical_feature_set: set[str]) -> str:
    if column_name in derived_categorical_feature_set:
        return 'derived_one_hot'
    if column_name.endswith('__change_count') or column_name.endswith('__nunique'):
        return 'control_context'
    if column_name in {'hour_of_day', 'day_of_week', 'day_of_year', 'month'}:
        return 'time_context'
    if column_name.endswith('_sin') or column_name.endswith('_cos'):
        return 'cyclic_time'
    if column_name in {'is_weekend', 'is_heating_season'}:
        return 'time_context'
    if column_name.startswith('days_since_last_') or column_name.startswith('post_') or column_name == 'recent_regime_change_flag':
        return 'event_context'
    return 'sensor_numeric'


metadata_columns = [column for column in strict_windows.columns if column in METADATA_ROLES]
manufacturer_values = sorted(selection_stats_windows['manufacturer'].dropna().unique().tolist())

derived_feature_columns = []
categorical_feature_map_rows = []
categorical_source_columns = [column for column in ['manufacturer', 'configuration_type', 'season_bucket'] if column in selection_stats_windows.columns]
categorical_source_columns.extend(
    sorted([column for column in selection_stats_windows.columns if column.endswith('__dominant')])
)

for column in categorical_source_columns:
    train_categories = (
        selection_stats_windows[column]
        .fillna('missing')
        .astype(str)
        .str.strip()
        .replace('', 'missing')
    )
    categories = sorted(train_categories.unique().tolist())
    if len(categories) <= 1 or len(categories) > MAX_ONE_HOT_CARDINALITY:
        continue

    for category in categories:
        feature_name = f'{column}__is__{sanitize_feature_name(category)}'
        derived_feature_columns.append(feature_name)
        for frame in [strict_windows, relaxed_windows, selection_stats_windows]:
            frame[feature_name] = (
                frame[column]
                .fillna('missing')
                .astype(str)
                .str.strip()
                .replace('', 'missing')
                .eq(category)
                .astype('int8')
            )
        categorical_feature_map_rows.append({
            'source_column': column,
            'category_value': category,
            'derived_feature_column': feature_name,
        })

categorical_feature_map_table = pd.DataFrame(categorical_feature_map_rows)
derived_categorical_feature_set = set(derived_feature_columns)
candidate_columns = [column for column in selection_stats_windows.columns if column not in metadata_columns]

feature_rows = []
for column in candidate_columns:
    series = selection_stats_windows[column]
    dtype_name = str(series.dtype)
    missing_rate = float(series.isna().mean())
    unique_count = int(series.nunique(dropna=False))
    is_numeric_or_bool = pd.api.types.is_numeric_dtype(series) or pd.api.types.is_bool_dtype(series)
    feature_family = detect_feature_family(column, derived_categorical_feature_set)

    row = {
        'column_name': column,
        'dtype': dtype_name,
        'feature_family': feature_family,
        'selection_split_column': FEATURE_SELECTION_SPLIT_COLUMN,
        'selection_split_value': FEATURE_SELECTION_SPLIT_VALUE,
        'missing_rate_train': missing_rate,
        'unique_count_train': unique_count,
        'selected_for_baseline': True,
        'exclusion_reason': '',
    }

    manufacturer_non_null_rates = {}
    for manufacturer in manufacturer_values:
        non_null_rate = float(
            selection_stats_windows.loc[
                selection_stats_windows['manufacturer'] == manufacturer, column
            ].notna().mean()
        )
        manufacturer_non_null_rates[manufacturer] = non_null_rate
        row[manufacturer.replace(' ', '_') + '_non_null_rate_train'] = non_null_rate

    exclusion_reasons = []
    if not is_numeric_or_bool:
        exclusion_reasons.append('non_numeric_or_bool')
    if missing_rate > MAX_FEATURE_MISSING_RATE:
        exclusion_reasons.append(f'missing_rate_gt_{MAX_FEATURE_MISSING_RATE:.2f}')
    if unique_count <= 1:
        exclusion_reasons.append('constant_feature')
    if REQUIRE_NON_NULL_IN_BOTH_MANUFACTURERS and any(rate == 0.0 for rate in manufacturer_non_null_rates.values()):
        exclusion_reasons.append('missing_in_one_manufacturer')

    if exclusion_reasons:
        row['selected_for_baseline'] = False
        row['exclusion_reason'] = '|'.join(exclusion_reasons)

    feature_rows.append(row)

feature_columns_table = pd.DataFrame(feature_rows).sort_values(
    ['selected_for_baseline', 'feature_family', 'missing_rate_train', 'column_name'],
    ascending=[False, True, True, True],
).reset_index(drop=True)

selected_feature_columns = feature_columns_table.loc[
    feature_columns_table['selected_for_baseline'], 'column_name'
].tolist()

feature_family_summary_table = (
    feature_columns_table
    .assign(selection_status=feature_columns_table['selected_for_baseline'].map({True: 'selected', False: 'excluded'}))
    .groupby(['feature_family', 'selection_status'], dropna=False)
    .size()
    .rename('feature_count')
    .reset_index()
    .sort_values(['feature_family', 'selection_status'])
)

feature_selection_summary = pd.DataFrame([
    {'??': 'feature ?? ?', '?': len(candidate_columns)},
    {'??': '?? feature ?', '?': len(selected_feature_columns)},
    {'??': 'metadata ?? ?', '?': len(metadata_columns)},
    {'??': 'one-hot feature ?', '?': len(derived_feature_columns)},
    {'??': 'strict ? ?', '?': len(strict_windows)},
    {'??': 'relaxed ? ?', '?': len(relaxed_windows)},
    {'??': 'feature ?? ?? ? ?', '?': len(selection_stats_windows)},
])

display(feature_selection_summary)
display(feature_family_summary_table)
display(feature_columns_table.head(30))


,??,?
0,feature ?? ?,259
1,?? feature ?,195
2,metadata ?? ?,37
3,one-hot feature ?,44
4,strict ? ?,2526
5,relaxed ? ?,2633
6,feature ?? ?? ? ?,1668


,feature_family,selection_status,feature_count
0,control_context,excluded,22
1,cyclic_time,selected,6
2,derived_one_hot,selected,44
3,event_context,excluded,4
4,event_context,selected,2
5,sensor_numeric,excluded,38
6,sensor_numeric,selected,137
7,time_context,selected,6


,column_name,dtype,feature_family,selection_split_column,selection_split_value,missing_rate_train,unique_count_train,selected_for_baseline,exclusion_reason,manufacturer_1_non_null_rate_train,manufacturer_2_non_null_rate_train
0,dow_cos,float64,cyclic_time,split_regime_based,train,0.0,7,True,,1.0,1.0
1,dow_sin,float64,cyclic_time,split_regime_based,train,0.0,7,True,,1.0,1.0
2,doy_cos,float64,cyclic_time,split_regime_based,train,0.0,216,True,,1.0,1.0
3,doy_sin,float64,cyclic_time,split_regime_based,train,0.0,209,True,,1.0,1.0
4,hour_cos,float64,cyclic_time,split_regime_based,train,0.0,4,True,,1.0,1.0
5,hour_sin,float64,cyclic_time,split_regime_based,train,0.0,3,True,,1.0,1.0
6,configuration_type__is__missing,int8,derived_one_hot,split_regime_based,train,0.0,2,True,,1.0,1.0
7,configuration_type__is__sh,int8,derived_one_hot,split_regime_based,train,0.0,2,True,,1.0,1.0
8,configuration_type__is__sh_dhw,int8,derived_one_hot,split_regime_based,train,0.0,2,True,,1.0,1.0
9,configuration_type__is__sh_dhw_with_sub_circuits,int8,derived_one_hot,split_regime_based,train,0.0,2,True,,1.0,1.0


In [6]:
exclusion_summary = (
    feature_columns_table.assign(
        exclusion_reason=feature_columns_table['exclusion_reason'].replace('', 'selected')
    )
    .groupby('exclusion_reason', dropna=False)
    .size()
    .rename('feature_count')
    .reset_index()
    .sort_values(['feature_count', 'exclusion_reason'], ascending=[False, True])
)

display(exclusion_summary)
display(feature_columns_table.loc[~feature_columns_table['selected_for_baseline']].head(30))


,exclusion_reason,feature_count
3,selected,195
2,missing_rate_gt_0.50|missing_in_one_manufacturer,40
0,constant_feature,14
1,missing_rate_gt_0.50,10


,column_name,dtype,feature_family,selection_split_column,selection_split_value,missing_rate_train,unique_count_train,selected_for_baseline,exclusion_reason,manufacturer_1_non_null_rate_train,manufacturer_2_non_null_rate_train
195,s_hc1_heating_pump_status_setpoint__change_count,float64,control_context,split_regime_based,train,0.521583,4,False,missing_rate_gt_0.50|missing_in_one_manufacturer,0.0,1.000000
196,s_hc1_heating_pump_status_setpoint__nunique,float64,control_context,split_regime_based,train,0.521583,3,False,missing_rate_gt_0.50|missing_in_one_manufacturer,0.0,1.000000
197,s_hc1_control_unit_mode__change_count,float64,control_context,split_regime_based,train,0.596523,3,False,missing_rate_gt_0.50|missing_in_one_manufacturer,0.0,0.843358
198,s_hc1_control_unit_mode__nunique,float64,control_context,split_regime_based,train,0.596523,4,False,missing_rate_gt_0.50|missing_in_one_manufacturer,0.0,0.843358
199,s_dhw_control_unit_mode__change_count,float64,control_context,split_regime_based,train,0.628297,2,False,missing_rate_gt_0.50|missing_in_one_manufacturer,0.0,0.776942
200,s_dhw_control_unit_mode__nunique,float64,control_context,split_regime_based,train,0.628297,3,False,missing_rate_gt_0.50|missing_in_one_manufacturer,0.0,0.776942
201,s_dhw_3-way_valve_status__change_count,float64,control_context,split_regime_based,train,0.637890,22,False,missing_rate_gt_0.50|missing_in_one_manufacturer,0.0,0.756892
202,s_dhw_3-way_valve_status__nunique,float64,control_context,split_regime_based,train,0.637890,3,False,missing_rate_gt_0.50|missing_in_one_manufacturer,0.0,0.756892
203,s_hc1.2_heating_pump_status__change_count,float64,control_context,split_regime_based,train,0.923261,4,False,missing_rate_gt_0.50|missing_in_one_manufacturer,0.0,0.160401
204,s_hc1.2_heating_pump_status__nunique,float64,control_context,split_regime_based,train,0.923261,3,False,missing_rate_gt_0.50|missing_in_one_manufacturer,0.0,0.160401


## 4. train-only 결측 대체값 계산

결측 대체 계약은 strict train split에서만 계산한다.
- numeric feature: median
- bool feature: mode


In [7]:
imputation_rows = []
for column in selected_feature_columns:
    series = selection_stats_windows[column]
    if pd.api.types.is_bool_dtype(series):
        non_null_series = series.dropna()
        if non_null_series.empty:
            fill_value = False
        else:
            fill_value = bool(non_null_series.mode(dropna=True).iloc[0])
        strategy = 'mode'
    else:
        fill_value = float(series.median()) if series.notna().any() else 0.0
        strategy = 'median'

    imputation_rows.append({
        'column_name': column,
        'imputation_strategy': strategy,
        'imputation_value': fill_value,
        'selection_split_column': FEATURE_SELECTION_SPLIT_COLUMN,
        'selection_split_value': FEATURE_SELECTION_SPLIT_VALUE,
    })

imputation_values_table = pd.DataFrame(imputation_rows)
display(imputation_values_table.head(20))


,column_name,imputation_strategy,imputation_value,selection_split_column,selection_split_value
0,dow_cos,median,-0.222521,split_regime_based,train
1,dow_sin,median,0.0,split_regime_based,train
2,doy_cos,median,0.5,split_regime_based,train
3,doy_sin,median,0.085731,split_regime_based,train
4,hour_cos,median,0.707107,split_regime_based,train
5,hour_sin,median,-0.707107,split_regime_based,train
6,configuration_type__is__missing,median,0.0,split_regime_based,train
7,configuration_type__is__sh,median,0.0,split_regime_based,train
8,configuration_type__is__sh_dhw,median,1.0,split_regime_based,train
9,configuration_type__is__sh_dhw_with_sub_circuits,median,0.0,split_regime_based,train


## 5. raw / imputed 학습 입력 저장

저장 산출물:
- strict raw
- relaxed raw
- strict imputed
- relaxed imputed
- 기본 trainable dataset = strict imputed


In [8]:

def build_dataset(source_df: pd.DataFrame, apply_imputation: bool) -> pd.DataFrame:
    dataset = source_df[metadata_columns + selected_feature_columns].copy()
    if apply_imputation:
        fill_map = dict(zip(imputation_values_table['column_name'], imputation_values_table['imputation_value']))
        for column in selected_feature_columns:
            dataset[column] = dataset[column].fillna(fill_map[column])
    return dataset

trainable_windows_strict = build_dataset(strict_windows, apply_imputation=False)
trainable_windows_relaxed = build_dataset(relaxed_windows, apply_imputation=False)
trainable_windows_strict_imputed = build_dataset(strict_windows, apply_imputation=True)
trainable_windows_relaxed_imputed = build_dataset(relaxed_windows, apply_imputation=True)
default_trainable_windows = trainable_windows_strict_imputed.copy()

trainable_windows_strict.to_csv(STRICT_RAW_PATH, index=False)
trainable_windows_relaxed.to_csv(RELAXED_RAW_PATH, index=False)
trainable_windows_strict_imputed.to_csv(STRICT_IMPUTED_PATH, index=False)
trainable_windows_relaxed_imputed.to_csv(RELAXED_IMPUTED_PATH, index=False)
default_trainable_windows.to_csv(DEFAULT_TRAINABLE_WINDOWS_PATH, index=False)
feature_columns_table.to_csv(FEATURE_COLUMNS_PATH, index=False)
metadata_columns_table.to_csv(METADATA_COLUMNS_PATH, index=False)
imputation_values_table.to_csv(IMPUTATION_VALUES_PATH, index=False)
categorical_feature_map_table.to_csv(CATEGORICAL_FEATURE_MAP_PATH, index=False)
feature_family_summary_table.to_csv(FEATURE_FAMILY_SUMMARY_PATH, index=False)

print('?? ?? ??:')
print(DEFAULT_TRAINABLE_WINDOWS_PATH)
print(STRICT_RAW_PATH)
print(RELAXED_RAW_PATH)
print(STRICT_IMPUTED_PATH)
print(RELAXED_IMPUTED_PATH)
print(FEATURE_COLUMNS_PATH)
print(METADATA_COLUMNS_PATH)
print(IMPUTATION_VALUES_PATH)
print(CATEGORICAL_FEATURE_MAP_PATH)
print(FEATURE_FAMILY_SUMMARY_PATH)


?? ?? ??:
C:\Project3\HeatGrid_Agent\data\processed\ml_features\trainable_windows.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_features\trainable_windows_strict.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_features\trainable_windows_relaxed.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_features\trainable_windows_strict_imputed.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_features\trainable_windows_relaxed_imputed.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_features\feature_columns.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_features\metadata_columns.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_features\imputation_values.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_features\categorical_feature_map.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_features\feature_family_summary.csv


In [9]:
preview_metadata = [
    column
    for column in [
        'manufacturer',
        'substation_id',
        'window_start',
        'window_end',
        'label',
        'fault_label',
        'estimated_lead_time_hours',
        'split_time_based',
        'split_substation_based',
    ]
    if column in default_trainable_windows.columns
]
preview_columns = preview_metadata + selected_feature_columns[:12]

display(default_trainable_windows[preview_columns].head())
display(feature_columns_table.loc[feature_columns_table['selected_for_baseline']].head(20))

missing_after_imputation = default_trainable_windows[selected_feature_columns].isna().sum().sum()
print('기본 학습 feature 잔여 결측 수:', int(missing_after_imputation))
print('기본 학습 입력 크기:', default_trainable_windows.shape)


,manufacturer,substation_id,window_start,window_end,label,fault_label,estimated_lead_time_hours,split_time_based,split_substation_based,dow_cos,dow_sin,doy_cos,doy_sin,hour_cos,hour_sin,configuration_type__is__missing,configuration_type__is__sh,configuration_type__is__sh_dhw,configuration_type__is__sh_dhw_with_sub_circuits,configuration_type__is__sh_with_buffer_tank,configuration_type__is__sh_with_sub_circuits
0,manufacturer 1,1,2019-12-01 00:00:00,2019-12-01 06:00:00,normal,NaN,NaN,validation,holdout,0.62349,-0.781831,0.861702,-0.507415,0.707107,0.707107,0,0,1,0,0,0
1,manufacturer 1,1,2019-12-01 06:00:00,2019-12-01 12:00:00,normal,NaN,NaN,validation,holdout,0.62349,-0.781831,0.861702,-0.507415,-0.707107,0.707107,0,0,1,0,0,0
3,manufacturer 1,1,2019-12-01 18:00:00,2019-12-02 00:00:00,normal,NaN,NaN,validation,holdout,0.62349,-0.781831,0.861702,-0.507415,0.707107,-0.707107,0,0,1,0,0,0
6,manufacturer 1,1,2019-12-02 12:00:00,2019-12-02 18:00:00,normal,NaN,NaN,validation,holdout,1.00000,0.000000,0.870285,-0.492548,-0.707107,-0.707107,0,0,1,0,0,0
7,manufacturer 1,1,2019-12-02 18:00:00,2019-12-03 00:00:00,normal,NaN,NaN,validation,holdout,1.00000,0.000000,0.870285,-0.492548,0.707107,-0.707107,0,0,1,0,0,0


,column_name,dtype,feature_family,selection_split_column,selection_split_value,missing_rate_train,unique_count_train,selected_for_baseline,exclusion_reason,manufacturer_1_non_null_rate_train,manufacturer_2_non_null_rate_train
0,dow_cos,float64,cyclic_time,split_regime_based,train,0.0,7,True,,1.0,1.0
1,dow_sin,float64,cyclic_time,split_regime_based,train,0.0,7,True,,1.0,1.0
2,doy_cos,float64,cyclic_time,split_regime_based,train,0.0,216,True,,1.0,1.0
3,doy_sin,float64,cyclic_time,split_regime_based,train,0.0,209,True,,1.0,1.0
4,hour_cos,float64,cyclic_time,split_regime_based,train,0.0,4,True,,1.0,1.0
5,hour_sin,float64,cyclic_time,split_regime_based,train,0.0,3,True,,1.0,1.0
6,configuration_type__is__missing,int8,derived_one_hot,split_regime_based,train,0.0,2,True,,1.0,1.0
7,configuration_type__is__sh,int8,derived_one_hot,split_regime_based,train,0.0,2,True,,1.0,1.0
8,configuration_type__is__sh_dhw,int8,derived_one_hot,split_regime_based,train,0.0,2,True,,1.0,1.0
9,configuration_type__is__sh_dhw_with_sub_circuits,int8,derived_one_hot,split_regime_based,train,0.0,2,True,,1.0,1.0


기본 학습 feature 잔여 결측 수: 0
기본 학습 입력 크기: (2526, 232)
